## Clase 2 - Exploracion y preparacion de datos
ENIGH 2024, hogares de Chiapas. 8 ejercicios cortos que conectan teoria (tipos, descriptiva, distancias, calidad, transformacion) con un dataset real.

In [2]:
from __future__ import annotations
from pathlib import Path
import math

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.metrics import pairwise_distances

from mineria_utils import (
        NAVY, TERRA, SAND, INK, PLOTLY_TEMPLATE, PLOTLY_FONT,
        clasificar_atributos, detectar_outliers_iqr, chi_square,
        aplicar_normalizaciones, distancia_matriz, imputar_por_grupo,
    )

# Paleta y estilo (consistente con las slides del curso)
NAVY = "#1C3257"
TERRA = "#AA4B37"
SAND = "#F4EFE6"
INK = "#1A1A1A"
PLOTLY_TEMPLATE = "plotly_white"
PLOTLY_FONT = dict(family="Helvetica, Arial, sans-serif", color=INK, size=13)

In [3]:
CSV_PATH = "./data/enigh_chiapas_2024.csv"
if not Path(CSV_PATH).exists():
    CSV_PATH = "../data/enigh_chiapas_2024.csv"
df = pd.read_csv(CSV_PATH)
print(f"Shape: {df.shape}")
df.head(3)

Shape: (2220, 20)


,folioviv,ubica_geo,clase_hog,tam_loc,est_socio,educa_jefe,sexo_jefe,edad_jefe,tot_integ,hombres,mujeres,mayores,menores,ing_cor,ingtrab,gasto_mon,alimentos,educa_espa,salud,transporte
0,702448201,7002,2,2,1,1,1,63,2,1,1,2,0,16336.33,4533.62,15678.43,7727.00,0.0,1418.46,4079.40
1,702448201,7002,2,2,1,1,1,54,2,1,1,2,0,5097.64,4705.71,13443.13,6878.47,0.0,0.00,4905.14
2,702448201,7002,2,2,1,7,1,22,3,1,2,2,1,42824.14,42824.14,33940.40,15441.22,1451.6,0.00,5719.48


### 1. Clasificar atributos por tipo
El dtype de pandas no es el tipo conceptual. `folioviv` y `ubica_geo` son `int64` pero son identificadores nominales; `tam_loc` y `est_socio` son enteros pero ordinales.

In [ ]:
HINTS = {
    "folioviv": "nominal",
    "ubica_geo": "nominal",
    "clase_hog": "nominal",
    "tam_loc": "ordinal",
    "est_socio": "ordinal",
    "educa_jefe": "ordinal",
    "sexo_jefe": "binario",
}
clasificar_atributos(df, hints=HINTS)

### 2. Descriptiva y la trampa de la media
Regla de Han para diagnosticar asimetria: `media - moda` ~ `3 * (media - mediana)`. En distribuciones de ingreso esperamos sesgo positivo fuerte.

In [4]:
num_cols = ["edad_jefe", "tot_integ", "ing_cor", "gasto_mon", "alimentos", "salud", "transporte"]
df[num_cols].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
edad_jefe,2220.0,50.68,16.06,17.00,38.00,50.00,62.00,95.00
tot_integ,2220.0,3.79,1.95,1.00,2.00,4.00,5.00,20.00
ing_cor,2220.0,40030.94,33217.20,1200.00,19504.55,30857.10,49197.04,373875.14
gasto_mon,2220.0,26431.17,19463.38,107.41,13539.32,21549.88,34312.54,214389.34
alimentos,2220.0,12381.49,8313.97,0.00,6733.84,10684.20,15839.86,93252.62
salud,2220.0,940.85,2843.80,0.00,0.00,156.52,792.37,51006.46
transporte,2220.0,4113.37,5028.67,0.00,870.96,2571.40,5459.09,46056.59


### 2.1 Tendencia central paso a paso
Antes de leer la salida de `df.describe()` como caja negra, recordemos las definiciones:

- **Media**: $\bar{x} = \frac{1}{n} \sum_{i=1}^{n} x_i$  -> suma de valores entre el numero de observaciones.
- **Mediana**: el valor central tras ordenar los datos (promedio de los dos centrales si $n$ es par).
- **Moda**: el valor con mayor frecuencia.

Las calculamos a mano sobre `ing_cor` y verificamos contra `pandas`.

In [5]:
ic = df["ing_cor"].dropna().reset_index(drop=True)
n = len(ic)

# Media: suma de valores / n
media_manual = ic.sum() / n

# Mediana: valor central tras ordenar (promedio de los dos centrales si n es par)
ic_ord = ic.sort_values().reset_index(drop=True)
if n % 2 == 1:
    mediana_manual = ic_ord.iloc[n // 2]
else:
    mediana_manual = (ic_ord.iloc[n // 2 - 1] + ic_ord.iloc[n // 2]) / 2

# Moda: valor con mayor frecuencia
conteos = ic.value_counts()
moda_manual = conteos.index[0]

print(f"n hogares: {n:,}")
print(f"Media    manual: {media_manual:>12,.2f}   pandas: {ic.mean():>12,.2f}")
print(f"Mediana  manual: {mediana_manual:>12,.2f}   pandas: {ic.median():>12,.2f}")
print(f"Moda     manual: {moda_manual:>12,.2f}   pandas: {ic.mode().iloc[0]:>12,.2f}")

n hogares: 2,220
Media    manual:    40,030.94   pandas:    40,030.94
Mediana  manual:    30,857.10   pandas:    30,857.10
Moda     manual:    20,511.91   pandas:    20,511.91


### 2.2 Varianza y desviacion estandar paso a paso
**Varianza muestral**: $s^2 = \frac{1}{n-1} \sum_{i=1}^{n} (x_i - \bar{x})^2$. **Desviacion estandar**: $s = \sqrt{s^2}$. El divisor $(n-1)$ — correccion de Bessel — es lo que usa pandas por defecto (`ddof=1`); `numpy.var/std` usan `ddof=0`.

In [6]:
desviaciones = ic - media_manual
suma_cuad = (desviaciones ** 2).sum()

var_manual = suma_cuad / (n - 1)       # varianza muestral (ddof=1)
std_manual = math.sqrt(var_manual)

print(f"Varianza  manual: {var_manual:>18,.2f}   pandas: {ic.var():>18,.2f}")
print(f"Std       manual: {std_manual:>18,.2f}   pandas: {ic.std():>18,.2f}")
# Nota: pd.Series.var/std usan ddof=1 por defecto; numpy usa ddof=0.

Varianza  manual:   1,103,382,162.86   pandas:   1,103,382,162.86
Std       manual:          33,217.20   pandas:          33,217.20


### 2.3 Cuartiles e IQR paso a paso
Q1 es el valor por debajo del cual cae el 25% de los datos; Q3, el 75%. El **IQR = Q3 - Q1** mide la dispersion del 50% central y es la base de la regla 1.5*IQR del proximo ejercicio. Los calculamos por **interpolacion lineal** sobre los datos ordenados (es la convencion que usan `numpy` y `pandas`).

In [7]:
def percentil_lineal(serie_ordenada: pd.Series, p: float) -> float:
    """Percentil p (en [0, 1]) por interpolacion lineal - mismo metodo que np.quantile."""
    n_loc = len(serie_ordenada)
    pos = p * (n_loc - 1)            # posicion fraccional en el array ordenado
    bajo = int(math.floor(pos))
    alto = int(math.ceil(pos))
    if bajo == alto:
        return float(serie_ordenada.iloc[bajo])
    frac = pos - bajo
    return float(serie_ordenada.iloc[bajo] * (1 - frac) + serie_ordenada.iloc[alto] * frac)


q1_manual = percentil_lineal(ic_ord, 0.25)
q2_manual = percentil_lineal(ic_ord, 0.50)
q3_manual = percentil_lineal(ic_ord, 0.75)
iqr_manual = q3_manual - q1_manual

print(f"Q1   manual: {q1_manual:>12,.2f}   pandas: {ic.quantile(0.25):>12,.2f}")
print(f"Q2   manual: {q2_manual:>12,.2f}   pandas: {ic.quantile(0.50):>12,.2f}")
print(f"Q3   manual: {q3_manual:>12,.2f}   pandas: {ic.quantile(0.75):>12,.2f}")
print(f"IQR  manual: {iqr_manual:>12,.2f}   pandas: {ic.quantile(0.75) - ic.quantile(0.25):>12,.2f}")
print(f"\nLimite inferior (Q1 - 1.5*IQR): {q1_manual - 1.5 * iqr_manual:,.2f}")
print(f"Limite superior (Q3 + 1.5*IQR): {q3_manual + 1.5 * iqr_manual:,.2f}")

Q1   manual:    19,504.55   pandas:    19,504.55
Q2   manual:    30,857.10   pandas:    30,857.10
Q3   manual:    49,197.03   pandas:    49,197.04
IQR  manual:    29,692.48   pandas:    29,692.49

Limite inferior (Q1 - 1.5*IQR): -25,034.18
Limite superior (Q3 + 1.5*IQR): 93,735.76


In [8]:
# `ic` ya fue definido en la celda 2.1; aqui lo reusamos para enfocarnos en la asimetria.
print(f"Media:    {ic.mean():>12,.0f} MXN")
print(f"Mediana:  {ic.median():>12,.0f} MXN")
print(f"Moda:     {ic.mode().iloc[0]:>12,.0f} MXN")
print(f"Diferencia media vs mediana: {(ic.mean() - ic.median()) / ic.median():.1%}")

Media:          40,031 MXN
Mediana:        30,857 MXN
Moda:           20,512 MXN
Diferencia media vs mediana: 29.7%


In [9]:
fig = px.histogram(df, x="ing_cor", nbins=60, template=PLOTLY_TEMPLATE,
                   color_discrete_sequence=[NAVY])
fig.add_vline(x=ic.mean(),   line_color=TERRA, line_width=2, annotation_text="media")
fig.add_vline(x=ic.median(), line_color=NAVY,  line_width=2, line_dash="dash", annotation_text="mediana")
fig.update_layout(title="Ingreso corriente trimestral - hogares de Chiapas",
                  font=PLOTLY_FONT, width=900, height=450)
fig.show()

### 3. Outliers con la regla 1.5 IQR
Marcamos como outlier todo valor fuera de `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`. Detectar no significa borrar.

In [10]:
mask = detectar_outliers_iqr(df["ing_cor"])
n_out = mask.sum()
print(f"Outliers en ing_cor: {n_out:,} de {len(df):,} hogares ({n_out/len(df):.1%})")

fig = px.box(df, y="ing_cor", points="outliers", template=PLOTLY_TEMPLATE,
             color_discrete_sequence=[NAVY])
fig.update_layout(title="ing_cor - distribucion y outliers (regla 1.5 IQR)",
                  font=PLOTLY_FONT, width=600, height=550)
fig.show()

Outliers en ing_cor: 130 de 2,220 hogares (5.9%)


### 4. Correlacion numerica y chi-cuadrado nominal
Pearson mide asociacion lineal entre numericas. Para nominales necesitamos chi-cuadrado de independencia sobre la tabla de contingencia.

In [11]:
corr_cols = ["edad_jefe", "tot_integ", "ing_cor", "ingtrab", "gasto_mon", "alimentos", "salud", "transporte"]
corr = df[corr_cols].corr().round(2)
fig = px.imshow(corr, text_auto=True,
                color_continuous_scale=[(0, SAND), (0.5, "white"), (1, NAVY)],
                zmin=-1, zmax=1, template=PLOTLY_TEMPLATE)
fig.update_layout(title="Correlacion Pearson", font=PLOTLY_FONT, width=700, height=600)
fig.show()

In [12]:
res1 = chi_square(df, "sexo_jefe", "clase_hog")
res2 = chi_square(df, "tam_loc",   "clase_hog")
print(f"sexo_jefe x clase_hog:  chi2 = {res1['chi2']:.2f},  p = {res1['p_value']:.4g},  dof = {res1['dof']}")
print(f"tam_loc   x clase_hog:  chi2 = {res2['chi2']:.2f},  p = {res2['p_value']:.4g},  dof = {res2['dof']}")
print("\nTabla de contingencia sexo_jefe x clase_hog:")
display(res1["observed"])

sexo_jefe x clase_hog:  chi2 = 130.34,  p = 3.288e-27,  dof = 4
tam_loc   x clase_hog:  chi2 = 18.66,  p = 0.09711,  dof = 12

Tabla de contingencia sexo_jefe x clase_hog:


clase_hog,1,2,3,4,5
sexo_jefe,,,,,
1,105,1168,374,10,3
2,94,251,211,3,1


### 5. Tres estrategias de imputacion
El CSV ya viene sin NaN, asi que introducimos un 10% artificial sobre `ing_cor` para comparar el efecto de cada estrategia sobre la varianza.

In [13]:
rng = np.random.default_rng(42)
df_miss = df.copy()
mask_miss = rng.random(len(df_miss)) < 0.10
df_miss.loc[mask_miss, "ing_cor"] = np.nan
print(f"Missing introducidos: {mask_miss.sum():,} ({mask_miss.mean():.1%})")

Missing introducidos: 210 (9.5%)


In [14]:
s_dropna = df_miss["ing_cor"].dropna()
s_median = df_miss["ing_cor"].fillna(df_miss["ing_cor"].median())
s_grupo  = imputar_por_grupo(df_miss, target="ing_cor", group="est_socio", estrategia="median")

In [16]:

fig = make_subplots(rows=1, cols=3, subplot_titles=("Drop NA", "Mediana global", "Mediana por est_socio"))
for i, (s, color) in enumerate(zip([s_dropna, s_median, s_grupo], [NAVY, TERRA, NAVY])):
    fig.add_trace(go.Histogram(x=s, nbinsx=50, marker_color=color, showlegend=False), row=1, col=i+1)
fig.update_layout(template=PLOTLY_TEMPLATE, font=PLOTLY_FONT, width=1200, height=400,
                  title="Comparacion de estrategias de imputacion")
fig.show()

print(f"Std original:           {df['ing_cor'].std():>12,.0f}")
print(f"Std tras dropna:        {s_dropna.std():>12,.0f}")
print(f"Std tras mediana glob:  {s_median.std():>12,.0f}  <- varianza colapsada")
print(f"Std tras mediana grupo: {s_grupo.std():>12,.0f}")

Std original:                 33,217
Std tras dropna:              33,474
Std tras mediana glob:        31,963  <- varianza colapsada
Std tras mediana grupo:       32,089


### 6. Tres normalizaciones
Min-max lleva a `[0, 1]`, z-score centra y escala por desviacion, decimal scaling divide por la potencia de 10 mas cercana al maximo.

In [17]:
norms = aplicar_normalizaciones(df["ing_cor"].dropna())
display(norms.describe().round(3))

fig = make_subplots(rows=1, cols=3, subplot_titles=("Min-Max [0,1]", "Z-score", "Decimal scaling"))
for i, col in enumerate(["minmax", "zscore", "decimal"]):
    fig.add_trace(go.Histogram(x=norms[col], nbinsx=50, marker_color=NAVY, showlegend=False), row=1, col=i+1)
fig.update_layout(template=PLOTLY_TEMPLATE, font=PLOTLY_FONT, width=1200, height=400,
                  title="Tres normalizaciones de ing_cor")
fig.show()

,original,minmax,zscore,decimal
count,2220.000,2220.000,2220.000,2220.000
mean,40030.937,0.104,0.000,0.040
std,33217.197,0.089,1.000,0.033
min,1200.000,0.000,-1.169,0.001
25%,19504.550,0.049,-0.618,0.020
50%,30857.100,0.080,-0.276,0.031
75%,49197.035,0.129,0.276,0.049
max,373875.140,1.000,10.050,0.374


### 7. Distancias entre hogares
Manhattan suma diferencias absolutas, Euclidean penaliza mas las diferencias grandes. La eleccion cambia el ranking de vecinos.

In [18]:
sample = df[corr_cols].dropna().sample(5, random_state=7).reset_index(drop=True)
sample_z = (sample - sample.mean()) / sample.std()

D_euc = distancia_matriz(sample_z, metrica="euclidean")
D_man = distancia_matriz(sample_z, metrica="manhattan")

fig = make_subplots(rows=1, cols=2, subplot_titles=("Euclidean", "Manhattan"))
fig.add_trace(go.Heatmap(z=D_euc, colorscale=[(0, SAND), (1, NAVY)], showscale=False,
                          text=np.round(D_euc, 2), texttemplate="%{text}"), row=1, col=1)
fig.add_trace(go.Heatmap(z=D_man, colorscale=[(0, SAND), (1, TERRA)], showscale=False,
                          text=np.round(D_man, 2), texttemplate="%{text}"), row=1, col=2)
fig.update_layout(template=PLOTLY_TEMPLATE, font=PLOTLY_FONT, width=1000, height=450,
                  title="Distancias pareja a pareja entre 5 hogares")
fig.show()

### 8. PCA 2D y visualizacion
PCA es una proyeccion lineal que maximiza varianza. `explained_variance_ratio_` nos dice cuanta informacion conservan los primeros componentes.

In [19]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

cols_pca = corr_cols
df_pca = df[cols_pca + ["est_socio"]].dropna()
X = StandardScaler().fit_transform(df_pca[cols_pca])
pca = PCA(n_components=2).fit(X)
pcs = pca.transform(X)

print(f"Varianza explicada por PC1: {pca.explained_variance_ratio_[0]:.1%}")
print(f"Varianza explicada por PC2: {pca.explained_variance_ratio_[1]:.1%}")
print(f"Acumulada (2 PC):           {pca.explained_variance_ratio_.sum():.1%}")

df_plot = df_pca.assign(PC1=pcs[:, 0], PC2=pcs[:, 1])
fig = px.scatter(df_plot, x="PC1", y="PC2", color="est_socio",
                 color_continuous_scale=[(0, TERRA), (1, NAVY)],
                 template=PLOTLY_TEMPLATE, opacity=0.6)
fig.update_layout(title="PCA 2D - coloreado por estrato socioeconomico",
                  font=PLOTLY_FONT, width=900, height=550)
fig.show()

Varianza explicada por PC1: 46.3%
Varianza explicada por PC2: 15.0%
Acumulada (2 PC):           61.3%


### Cierre - mapa de decision aplicado
- Tipo conceptual primero, dtype despues: `folioviv` numerica almacenada no implica numerica analizable.
- Si la distribucion es asimetrica (media >> mediana), reportar mediana y discutir cola, no recortar a ciegas.
- Outliers se investigan antes de borrarse; el 5-10% superior de `ing_cor` es senal, no ruido.
- Asociacion depende del tipo: Pearson para numerico-numerico, chi-cuadrado para nominal-nominal.
- Imputar global colapsa varianza; imputar por grupo preserva estructura. Normalizar no arregla sesgo.